In [ ]:
import pandas as pd
import tensorflow as tf

# Check if GPU is available
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

from sklearn.preprocessing import OneHotEncoder
import nltk
import spacy 

,OGC_FID,boamp_theme_boamp,boamp_libelle_annonce,boamp_statut_annonce,boamp_date_fin_de_marche,boamp_num_departement_diffusion,boamp_nom_acheteur,boamp_intitule,cal_theme_signalement,cal_réponse_signalement,cal_mot_rang_1,cal_mot_rang_2,cal_date_traitement,cal_num_departement_exec,cal_insee_commune_exec,cal_nom_commune_exec
0,9723,Espaces verts,Avis de marché,INITIAL,2024-02-02,31,TOULOUSE METROPOLE,Travaux d'aménagement de l'île du Ramier à Tou...,Route,Pris en compte,travaux,aménagement,2023-12-22,31,31555,TOULOUSE
1,10502,Station d'épuration (travaux),Avis de marché,INITIAL,2024-08-08,09,Syndicat Mixte Départemental de l'Eau et de l'...,Travaux de construction de la nouvelle station...,Points d'intérêt,Pris en compte,nouvelle,station,2024-05-30,9,09332,VERNIOLLE
2,10900,Voirie et réseaux divers,Rectificatif,RECTIFICATIF,2024-08-23,33,Bordeaux Métropole,"Aménagement de la place de la Renaudel, rues ...",Route,Pris en compte,aménagement,place,2024-08-14,33,33063,BORDEAUX
3,779,"Prestations de services, Délégation de service...",Avis de marché,INITIAL,2024-04-16,75,Ville de Paris,Rénovation du parc de stationnement St Martin ...,Points d'intérêt,Rejete (hors specs),transformation,parc,2023-12-29,75,75056,PARIS
4,5614,Génie civil,Avis de marché,INITIAL,2024-07-11,43,Département de la Haute-Loire,Rd12 - Reconstruction Du Pont Sur La Loire A B...,Route,Pris en compte,reconstruction,pont,2024-06-14,43,43020,BAS EN BASSET


In [3]:
import pandas as pd
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
import nltk
import spacy 

In [4]:
# Télécharger les stopwords français si nécessaire
nltk.download('stopwords')

# Initialiser le modèle spaCy pour la lemmatisation
nlp = spacy.load('fr_core_news_sm')

# Définir les colonnes textuelles et catégorielles
text_columns = ['boamp_theme_boamp', 'boamp_intitule', 'texte_annonce']
categorical_columns = ['cal_mot_rang_1', 'cal_mot_rang_2', 'boamp_libelle_annonce', 'boamp_statut_annonce', 'cal_theme_signalement', 'boamp_num_departement_diffusion']

# Ajouter des mots vides supplémentaires
additional_stop_words = set([
    "de","la", "le", "les", 
    "des", "du", "un", "une", "en", "et", 
    "à", "au", "aux", "par", "pour", "sur", 
    "dans", "avec", "ce", "cette", "ces", "mon", 
    "ma", "mes", "ton", "ta", "tes", "son", "sa", 
    "ses", "notre", "nos", "votre", "vos", "leur", 
    "leurs", "ne", "pas", "plus", "moins", "ou", 
    "si", "oui", "non", "je", "tu", "il", "elle", 
    "nous", "vous", "ils", "elles", "me", "te", 
    "se", "moi", "toi", "lui", "leur", "y", "en",
    "d'", "l'","s'", "c'", "n'", "j'", "t'", "m'", "qu'", "l'"    
])

stop_words = set(stopwords.words("french")).union(additional_stop_words)

# Fonction de nettoyage et de lemmatisation des textes
def clean_text(text):
    if isinstance(text, float):
        return ''

    # Suppression des caractères spéciaux et de la ponctuation (conservation des lettres avec accents et apostrophes)
    text = re.sub(r'[^\w\s\']', '', text)

    # Conversion en minuscule
    text = text.lower()

    # Traitement du texte avec spaCy pour lemmatisation
    doc = nlp(text)

    # Suppression des mots vides
    lemmatized_tokens = [token.lemma_ for token in doc if token.text not in stop_words]

    # Joindre les tokens en une seule chaîne
    cleaned_text = ' '.join(lemmatized_tokens)

    return cleaned_text

# Étape 1 : Appliquer le nettoyage et la lemmatisation aux colonnes textuelles
for col in text_columns:
    train_df[col] = train_df[col].apply(lambda x: clean_text(x))

# Étape 2 : Vectorisation des colonnes textuelles avec TF-IDF (sans réduction des dimensions)
vectorized_dfs = {}
for col in text_columns:
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))  # Inclut unigrams et bigrams
    vectorized_df = pd.DataFrame(
        vectorizer.fit_transform(train_df[col]).toarray(),
        columns=[f"{col}_{feat}" for feat in vectorizer.get_feature_names_out()]
    )
    vectorized_dfs[col] = vectorized_df

# Étape 3 : Encodage des colonnes catégorielles avec OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_encoded = pd.DataFrame(
    encoder.fit_transform(train_df[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

# Étape 4 : Concaténer toutes les colonnes vectorisées et encodées
concat1_df = pd.concat([*vectorized_dfs.values(), categorical_encoded], axis=1)

# Affichage d'un échantillon des données finales
print(concat1_df.head())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gauth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


   boamp_theme_boamp_abonnemer  boamp_theme_boamp_abonnemer equipemer  \
0                          0.0                                    0.0   
1                          0.0                                    0.0   
2                          0.0                                    0.0   
3                          0.0                                    0.0   
4                          0.0                                    0.0   

   boamp_theme_boamp_abri  boamp_theme_boamp_abri abonnemer  \
0                     0.0                               0.0   
1                     0.0                               0.0   
2                     0.0                               0.0   
3                     0.0                               0.0   
4                     0.0                               0.0   

   boamp_theme_boamp_abri bardage  boamp_theme_boamp_abri contrôle  \
0                             0.0                              0.0   
1                             0.0         

In [4]:
import pandas as pd
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
import nltk
import spacy 

In [12]:
# Télécharger les stopwords français si nécessaire
nltk.download('stopwords')

# Initialiser le modèle spaCy pour la lemmatisation
nlp = spacy.load('fr_core_news_sm')

# Définir les colonnes textuelles et catégorielles
text_columns = ['boamp_theme_boamp', 'boamp_intitule', 'texte_annonce']
categorical_columns = ['cal_mot_rang_1', 'cal_mot_rang_2', 'boamp_libelle_annonce', 'boamp_statut_annonce', 'cal_theme_signalement', 'boamp_num_departement_diffusion']

# Ajouter des mots vides supplémentaires
additional_stop_words = set([
    "de","la", "le", "les", 
    "des", "du", "un", "une", "en", "et", 
    "à", "au", "aux", "par", "pour", "sur", 
    "dans", "avec", "ce", "cette", "ces", "mon", 
    "ma", "mes", "ton", "ta", "tes", "son", "sa", 
    "ses", "notre", "nos", "votre", "vos", "leur", 
    "leurs", "ne", "pas", "plus", "moins", "ou", 
    "si", "oui", "non", "je", "tu", "il", "elle", 
    "nous", "vous", "ils", "elles", "me", "te", 
    "se", "moi", "toi", "lui", "leur", "y", "en",
    "d'", "l'","s'", "c'", "n'", "j'", "t'", "m'", "qu'", "l'"    
])

stop_words = set(stopwords.words("french")).union(additional_stop_words)

# Fonction de nettoyage et de lemmatisation des textes
def clean_text(text):
    if isinstance(text, float):
        return ''

    # Suppression des caractères spéciaux et de la ponctuation (conservation des lettres avec accents et apostrophes)
    text = re.sub(r'[^\w\s\']', '', text)

    # Conversion en minuscule
    text = text.lower()

    # Traitement du texte avec spaCy pour lemmatisation
    doc = nlp(text)

    # Suppression des mots vides
    lemmatized_tokens = [token.lemma_ for token in doc if token.text not in stop_words]

    # Joindre les tokens en une seule chaîne
    cleaned_text = ' '.join(lemmatized_tokens)

    return cleaned_text

# Étape 1 : Appliquer le nettoyage et la lemmatisation aux colonnes textuelles
for col in text_columns:
    train_df[col] = train_df[col].apply(lambda x: clean_text(x))

# Étape 2 : Vectorisation des colonnes textuelles avec TF-IDF (sans réduction des dimensions)
vectorized_dfs = {}
for col in text_columns:
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))  # Inclut unigrams et bigrams
    vectorized_df = pd.DataFrame(
        vectorizer.fit_transform(train_df[col]).toarray(),
        columns=[f"{col}_{feat}" for feat in vectorizer.get_feature_names_out()]
    )
    vectorized_dfs[col] = vectorized_df

# Étape 3 : Encodage des colonnes catégorielles avec OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_encoded = pd.DataFrame(
    encoder.fit_transform(train_df[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

# Étape 4 : Concaténer toutes les colonnes vectorisées et encodées
concat1_df = pd.concat([*vectorized_dfs.values(), categorical_encoded], axis=1)

# Affichage d'un échantillon des données finales
print(concat1_df.head())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gauth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\gauth\AppData\Local\Temp\ipykernel_20440\4016682768.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  concat2_df[col] = 0  # Ajouter les colonnes manquantes avec des zéros
C:\Users\gauth\AppData\Local\Temp\ipykernel_20440\4016682768.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  concat2_df[col] = 0  # Ajouter les colonnes manquantes avec des zéros
C:\Use

   boamp_theme_boamp_abonnement  boamp_theme_boamp_abonnemer  \
0                             0                            0   
1                             0                            0   
2                             0                            0   
3                             0                            0   
4                             0                            0   

   boamp_theme_boamp_abonnemer equipemer  boamp_theme_boamp_abri  \
0                                      0                     0.0   
1                                      0                     0.0   
2                                      0                     0.0   
3                                      0                     0.0   
4                                      0                     0.0   

   boamp_theme_boamp_abri abonnement  boamp_theme_boamp_abri bardage  \
0                                  0                               0   
1                                  0                          

In [5]:
# Enregistrer le DataFrame final en fichier CSV pour un usage futur
concat1_df.to_csv("final_dataframe.csv", index=False)

0       1
1       1
2       1
3       0
4       1
       ..
4995    1
4996    1
4997    1
4998    1
4999    0
Name: cal_réponse_signalement, Length: 5000, dtype: int64


In [44]:
final_df = pd.read_csv("final_dataframe.csv")
final_df.head()

C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: Logistic Regression
Accuracy: 0.81
Weighted F1 Score: 0.795
[[119 140]
 [ 51 690]]
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.46      0.55       259
           1       0.83      0.93      0.88       741

    accuracy                           0.81      1000
   macro avg       0.77      0.70      0.72      1000
weighted avg       0.80      0.81      0.79      1000

Cross-Validation Mean Accuracy: 0.76
--------------------------------------------------


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: K Nearest Neighbors
Accuracy: 0.75
Weighted F1 Score: 0.728
[[ 89 170]
 [ 84 657]]
Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.34      0.41       259
           1       0.79      0.89      0.84       741

    accuracy                           0.75      1000
   macro avg       0.65      0.62      0.63      1000
weighted avg       0.72      0.75      0.73      1000

Cross-Validation Mean Accuracy: 0.73
--------------------------------------------------
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Model: Neural Network
Accuracy: 0.76
Weighted F1 Score: 0.758
[[132 127]
 [113 628]]
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.51      0.52       259
           1       0.83      0.85      0.84       741

    accuracy                           0.76      1000
   macro avg       0.69      0.68      0.68      1000
weighted avg       0.76      0.76      0.76      1000

C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [5]:
# Transformation de la colonne cible 
train_df['cal_réponse_signalement'] = train_df['cal_réponse_signalement'].map({'Pris en compte': 1, 'Rejete (hors specs)': 0})
print(train_df['cal_réponse_signalement'])

0       1
1       1
2       1
3       0
4       1
       ..
4995    1
4996    1
4997    1
4998    1
4999    0
Name: cal_réponse_signalement, Length: 5000, dtype: int64


In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Séparation des données (caractéristiques et cible)
X = final_df 
y = train_df['cal_réponse_signalement']  # La variable cible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création et entraînement des modèles
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Ajout du modèle de réseau de neurones aux modèles existants
nn_model = create_nn_model(X_train.shape[1])
nn_model_name = 'Neural Network'
models[nn_model_name] = nn_model

# Évaluation des modèles
for name, model in models.items():
    if name == nn_model_name:
        # Entraînement du modèle de réseau de neurones
        model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
        y_pred = (model.predict(X_test) > 0.5).astype("int32")
    else:
        # Entraînement des autres modèles
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5) if name != nn_model_name else [accuracy]  # Placeholder for neural network

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.3f}")
    print(confusion_matrix(y_test, y_pred))
    print(f"Classification Report:\n{report}")
    if name != nn_model_name:
        print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: Logistic Regression
Accuracy: 0.81
Weighted F1 Score: 0.795
[[119 140]
 [ 51 690]]
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.46      0.55       259
           1       0.83      0.93      0.88       741

    accuracy                           0.81      1000
   macro avg       0.77      0.70      0.72      1000
weighted avg       0.80      0.81      0.79      1000

Cross-Validation Mean Accuracy: 0.76
--------------------------------------------------


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: K Nearest Neighbors
Accuracy: 0.75
Weighted F1 Score: 0.728
[[ 89 170]
 [ 84 657]]
Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.34      0.41       259
           1       0.79      0.89      0.84       741

    accuracy                           0.75      1000
   macro avg       0.65      0.62      0.63      1000
weighted avg       0.72      0.75      0.73      1000

Cross-Validation Mean Accuracy: 0.73
--------------------------------------------------
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Model: Neural Network
Accuracy: 0.76
Weighted F1 Score: 0.758
[[132 127]
 [113 628]]
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.51      0.52       259
           1       0.83      0.85      0.84       741

    accuracy                           0.76      1000
   macro avg       0.69      0.68      0.68      1000
weighted avg       0.76      0.76      0.76      1000

C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [45]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées
y = train_df['cal_réponse_signalement']  # Variable cible

# Plage de valeurs pour n_components à tester
n_components_range = list(range(50, 301, 50))
variance_threshold = 0.85  # Objectif de variance cumulative souhaitée (ajustable entre 0.80 et 0.90)
best_n_components = None
best_explained_variance = 0

# Recherche du meilleur n_components pour capturer la variance désirée
for n_components in n_components_range:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)

    # Calcul de la variance cumulée
    cumulative_variance = svd.explained_variance_ratio_.sum()
    
    if cumulative_variance >= variance_threshold and cumulative_variance > best_explained_variance:
        best_n_components = n_components
        best_explained_variance = cumulative_variance

print(f"\nMeilleure valeur pour n_components : {best_n_components} avec variance cumulée : {best_explained_variance:.4f}")

# Appliquer TruncatedSVD avec le meilleur n_components
svd = TruncatedSVD(n_components=best_n_components, random_state=42)
X_reduced = svd.fit_transform(X)

# Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
scaler = MinMaxScaler()
X_reduced = scaler.fit_transform(X_reduced)
final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(best_n_components)])

# Séparation des données (caractéristiques réduites et cible)
X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

# Création des modèles de classification
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Liste pour stocker les résultats du weighted F1 score par modèle
f1_scores = []

# Évaluation des modèles avec les données réduites
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted',pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Enregistrer le weighted F1 score pour la visualisation
    f1_scores.append(weighted_f1)

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.2f}")
    print(f"Classification Report:\n{report}")
    print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)

# Visualisation de l'évolution du weighted F1 score par classifieur
plt.figure(figsize=(10, 6))
plt.bar(models.keys(), f1_scores, color='skyblue')
plt.title('Weighted F1 Score par Classifieur')
plt.xlabel('Classifieurs')
plt.ylabel('Weighted F1 Score')
plt.xticks(rotation=45)
plt.show()


Meilleure valeur pour n_components : None avec variance cumulée : 0.0000


InvalidParameterError: The 'n_components' parameter of TruncatedSVD must be an int in the range [1, inf). Got None instead.

### Manuellement

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées
y = train_df['cal_réponse_signalement']  # Variable cible

# Plage de valeurs pour n_components à tester
n_components_range = list(range(50, 301, 50))
variance_threshold = 0.85  # Objectif de variance cumulative souhaitée (ajustable entre 0.80 et 0.90)
best_n_components = None
best_explained_variance = 0

# Recherche du meilleur n_components pour capturer la variance désirée
for n_components in n_components_range:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)

    # Calcul de la variance cumulée
    cumulative_variance = svd.explained_variance_ratio_.sum()
    
    if cumulative_variance >= variance_threshold and cumulative_variance > best_explained_variance:
        best_n_components = n_components
        best_explained_variance = cumulative_variance

print(f"\nMeilleure valeur pour n_components : {best_n_components} avec variance cumulée : {best_explained_variance:.4f}")

# Appliquer TruncatedSVD avec le meilleur n_components
svd = TruncatedSVD(n_components=best_n_components, random_state=42)
X_reduced = svd.fit_transform(X)

# Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
scaler = MinMaxScaler()
X_reduced = scaler.fit_transform(X_reduced)
final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(best_n_components)])

# Séparation des données (caractéristiques réduites et cible)
X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

# Création des modèles de classification
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Liste pour stocker les résultats du weighted F1 score par modèle
f1_scores = []

# Évaluation des modèles avec les données réduites
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted',pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Enregistrer le weighted F1 score pour la visualisation
    f1_scores.append(weighted_f1)

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.2f}")
    print(f"Classification Report:\n{report}")
    print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)

# Visualisation de l'évolution du weighted F1 score par classifieur
plt.figure(figsize=(10, 6))
plt.bar(models.keys(), f1_scores, color='skyblue')
plt.title('Weighted F1 Score par Classifieur')
plt.xlabel('Classifieurs')
plt.ylabel('Weighted F1 Score')
plt.xticks(rotation=45)
plt.show()

In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Séparation des données (caractéristiques et cible)
X = final_df 
y = train_df['cal_réponse_signalement']  # La variable cible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création et entraînement des modèles
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Ajout du modèle de réseau de neurones aux modèles existants
nn_model = create_nn_model(X_train.shape[1])
nn_model_name = 'Neural Network'
models[nn_model_name] = nn_model

# Évaluation des modèles
for name, model in models.items():
    if name == nn_model_name:
        # Entraînement du modèle de réseau de neurones
        model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
        y_pred = (model.predict(X_test) > 0.5).astype("int32")
    else:
        # Entraînement des autres modèles
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5) if name != nn_model_name else [accuracy]  # Placeholder for neural network

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.3f}")
    print(confusion_matrix(y_test, y_pred))
    print(f"Classification Report:\n{report}")
    if name != nn_model_name:
        print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: Logistic Regression
Accuracy: 0.81
Weighted F1 Score: 0.795
[[119 140]
 [ 51 690]]
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.46      0.55       259
           1       0.83      0.93      0.88       741

    accuracy                           0.81      1000
   macro avg       0.77      0.70      0.72      1000
weighted avg       0.80      0.81      0.79      1000

Cross-Validation Mean Accuracy: 0.76
--------------------------------------------------


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


Model: K Nearest Neighbors
Accuracy: 0.75
Weighted F1 Score: 0.728
[[ 89 170]
 [ 84 657]]
Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.34      0.41       259
           1       0.79      0.89      0.84       741

    accuracy                           0.75      1000
   macro avg       0.65      0.62      0.63      1000
weighted avg       0.72      0.75      0.73      1000

Cross-Validation Mean Accuracy: 0.73
--------------------------------------------------
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Model: Neural Network
Accuracy: 0.76
Weighted F1 Score: 0.758
[[132 127]
 [113 628]]
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.51      0.52       259
           1       0.83      0.85      0.84       741

    accuracy                           0.76      1000
   macro avg       0.69      0.68      0.68      1000
weighted avg       0.76      0.76      0.76      1000

C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Utiliser final_df, votre DataFrame vectorisé
X = final_train_df  # Les données vectorisées sans NaN
y = train_df['cal_réponse_signalement']  # Variable cible sans NaN

# Assurer que les dimensions de X et y correspondent
assert X.shape[0] == y.shape[0], "Les dimensions de X et y ne correspondent pas"

# Liste des valeurs de n_components à tester manuellement
n_components_to_test = [300, 325, 400, 500, 600]

# Dictionnaire pour stocker les scores pour chaque valeur de n_components
results = {model: [] for model in ['Logistic Regression', 'Neural Network']}

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Boucle pour tester chaque valeur de n_components
for n_components in n_components_to_test:
    print(f"Testing n_components={n_components}")
    
    # Appliquer TruncatedSVD
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)
    
    # Afficher la variance expliquée par chaque composante
    cumulative_variance = svd.explained_variance_ratio_.sum()
    print(f"Cumulative variance for n_components={n_components}: {cumulative_variance:.4f}")

    # Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
    scaler = MinMaxScaler()
    X_reduced = scaler.fit_transform(X_reduced)
    final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(n_components)])

    # Séparation des données (caractéristiques réduites et cible)
    X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

    # Création des modèles de classification
    models = {
        #'Logistic Regression': LogisticRegression(max_iter=1000),
        'Neural Network': create_nn_model(X_train.shape[1]),
    }

    # Évaluation des modèles avec les données réduites
    for name, model in models.items():
        if name == 'Neural Network':
            # Entraînement du modèle de réseau de neurones
            model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
            y_pred = (model.predict(X_test) > 0.5).astype("int32")
            accuracy = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
            results[name].append((n_components, weighted_f1))
        else:
            # Entraînement des autres modèles
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
            results[name].append((n_components, weighted_f1))
            cv_scores = cross_val_score(model, X_train, y_train, cv=5)
        
        # Affichage des résultats
        print(f"Model: {name}")
        print(f"Accuracy: {accuracy:.2f}")
        print(f"Weighted F1 Score: {weighted_f1:.3f}")
        if name != 'Neural Network':
            report = classification_report(y_test, y_pred, zero_division=0)
            print(f"Classification Report:\n{report}")
            print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
        print('-' * 50)

# Visualisation des résultats pour chaque modèle
plt.figure(figsize=(12, 8))
for model, scores in results.items():
    n_components, f1_scores = zip(*scores)
    plt.plot(n_components, f1_scores, marker='o', label=model)
    for x, y in zip(n_components, f1_scores):
        plt.text(x, y, f'{y:.3f}', fontsize=9, ha='right')

plt.title('Weighted F1 Score par Modèle en fonction de n_components')
plt.xlabel('n_components')
plt.ylabel('Weighted F1 Score')
plt.legend()
plt.grid(True)
plt.show()



In [67]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Séparation des données (caractéristiques et cible)
X = final_df 
y = train_df['cal_réponse_signalement']  # La variable cible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création et entraînement des modèles
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Ajout du modèle de réseau de neurones aux modèles existants
nn_model = create_nn_model(X_train.shape[1])
nn_model_name = 'Neural Network'
models[nn_model_name] = nn_model

# Évaluation des modèles
for name, model in models.items():
    if name == nn_model_name:
        # Entraînement du modèle de réseau de neurones
        model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
        y_pred = (model.predict(X_test) > 0.5).astype("int32")
    else:
        # Entraînement des autres modèles
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5) if name != nn_model_name else [accuracy]  # Placeholder for neural network

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.3f}")
    print(confusion_matrix(y_test, y_pred))
    print(f"Classification Report:\n{report}")
    if name != nn_model_name:
        print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Meilleurs hyperparamètres : {'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

Optimisation de la régression logistique :
Weighted F1 Score après optimisation : 0.787
Classification Report :
               precision    recall  f1-score   support

           0       0.67      0.46      0.54       259
           1       0.83      0.92      0.87       741

    accuracy                           0.80      1000
   macro avg       0.75      0.69      0.71      1000
weighted avg       0.79      0.80      0.79      1000

Fitting 4 folds for each of 486 candidates, totalling 1944 fits


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(
C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Meilleurs hyperparamètres pour le réseau de neurones : {'batch_size': 128, 'epochs': 30, 'model__dropout_rate': 0.2, 'model__learning_rate': 0.001, 'model__units': 128, 'optimizer': 'rmsprop'}

Optimisation du réseau de neurones :
Weighted F1 Score pour le réseau de neurones après optimisation : 0.707
Classification Report :
               precision    recall  f1-score   support

           0       0.76      0.16      0.26       259
           1       0.77      0.98      0.86       741

    accuracy                           0.77      1000
   macro avg       0.76      0.57      0.56      1000
weighted avg       0.77      0.77      0.71      1000



C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [45]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées
y = train_df['cal_réponse_signalement']  # Variable cible

# Plage de valeurs pour n_components à tester
n_components_range = list(range(50, 301, 50))
variance_threshold = 0.85  # Objectif de variance cumulative souhaitée (ajustable entre 0.80 et 0.90)
best_n_components = None
best_explained_variance = 0

# Recherche du meilleur n_components pour capturer la variance désirée
for n_components in n_components_range:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)

    # Calcul de la variance cumulée
    cumulative_variance = svd.explained_variance_ratio_.sum()
    
    if cumulative_variance >= variance_threshold and cumulative_variance > best_explained_variance:
        best_n_components = n_components
        best_explained_variance = cumulative_variance

print(f"\nMeilleure valeur pour n_components : {best_n_components} avec variance cumulée : {best_explained_variance:.4f}")

# Appliquer TruncatedSVD avec le meilleur n_components
svd = TruncatedSVD(n_components=best_n_components, random_state=42)
X_reduced = svd.fit_transform(X)

# Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
scaler = MinMaxScaler()
X_reduced = scaler.fit_transform(X_reduced)
final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(best_n_components)])

# Séparation des données (caractéristiques réduites et cible)
X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

# Création des modèles de classification
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Liste pour stocker les résultats du weighted F1 score par modèle
f1_scores = []

# Évaluation des modèles avec les données réduites
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted',pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Enregistrer le weighted F1 score pour la visualisation
    f1_scores.append(weighted_f1)

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.2f}")
    print(f"Classification Report:\n{report}")
    print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)

# Visualisation de l'évolution du weighted F1 score par classifieur
plt.figure(figsize=(10, 6))
plt.bar(models.keys(), f1_scores, color='skyblue')
plt.title('Weighted F1 Score par Classifieur')
plt.xlabel('Classifieurs')
plt.ylabel('Weighted F1 Score')
plt.xticks(rotation=45)
plt.show()


Meilleure valeur pour n_components : None avec variance cumulée : 0.0000


InvalidParameterError: The 'n_components' parameter of TruncatedSVD must be an int in the range [1, inf). Got None instead.

In [8]:
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from scikeras.wrappers import KerasClassifier
import numpy as np

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées sans NaN
y = train_df['cal_réponse_signalement']  # Variable cible sans NaN

# Assurer que les dimensions de X et y correspondent
assert X.shape[0] == y.shape[0], "Les dimensions de X et y ne correspondent pas"

# Appliquer TruncatedSVD avec n_components = 325
svd = TruncatedSVD(n_components=325, random_state=42)
X_reduced = svd.fit_transform(X)

# Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
scaler = MinMaxScaler()
X_reduced = scaler.fit_transform(X_reduced)
final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(325)])

# Séparation des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

# Définition de l'espace des hyperparamètres pour la régression logistique
param_grid = {
    'C': [1],  # Essai de différentes valeurs pour le paramètre de régularisation
    'solver': ['lbfgs'],  # Différents solveurs possibles pour optimiser
    'penalty': ['l2'],  # 'l1' est disponible uniquement pour le solver 'liblinear'
    'max_iter': [100]  # Tester différents nombres d'itérations
}

# Initialisation du modèle de régression logistique
log_reg = LogisticRegression()

# Utilisation de GridSearchCV pour la recherche d'hyperparamètres
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=2)

# Exécution de GridSearchCV
grid_search.fit(X_train, y_train)

# Affichage des meilleurs hyperparamètres
print("Meilleurs hyperparamètres :", grid_search.best_params_)

# Entraînement du modèle avec les meilleurs hyperparamètres
best_log_reg = grid_search.best_estimator_
best_log_reg.fit(X_train, y_train)
y_pred = best_log_reg.predict(X_test)

# Calcul du weighted F1 score et affichage du rapport de classification
weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
report = classification_report(y_test, y_pred, zero_division=0)

print("\nOptimisation de la régression logistique :")
print(f"Weighted F1 Score après optimisation : {weighted_f1:.3f}")
print("Classification Report :\n", report)

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(units=128, learning_rate=0.001, dropout_rate=0.5):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(units // 2, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Utiliser KerasClassifier wrapper
nn_model = KerasClassifier(model=create_nn_model, epochs=20, batch_size=32, verbose=0)

# Définir l'espace des hyperparamètres pour le réseau de neurones
param_grid_nn = {
    'model__units': [64, 128, 256],
    'model__learning_rate': [0.001, 0.01, 0.1],
    'model__dropout_rate': [0.2, 0.3, 0.5],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20, 30],
    'optimizer': ['adam', 'rmsprop']
}

# Utilisation de GridSearchCV pour la recherche d'hyperparamètres du réseau de neurones
grid_search_nn = GridSearchCV(estimator=nn_model, param_grid=param_grid_nn, cv=4, scoring='f1_weighted', n_jobs=-1, verbose=2)

# Exécution de GridSearchCV pour le réseau de neurones
grid_search_nn.fit(X_train, y_train)

# Affichage des meilleurs hyperparamètres pour le réseau de neurones
print("Meilleurs hyperparamètres pour le réseau de neurones :", grid_search_nn.best_params_)

# Entraînement du modèle de réseau de neurones avec les meilleurs hyperparamètres
best_nn_model = grid_search_nn.best_estimator_
best_nn_model.fit(X_train, y_train)
y_pred_nn = (best_nn_model.predict(X_test) > 0.5).astype("int32")

# Calcul du weighted F1 score et affichage du rapport de classification pour le réseau de neurones
weighted_f1_nn = f1_score(y_test, y_pred_nn, average='weighted', pos_label="Pris en compte")
report_nn = classification_report(y_test, y_pred_nn, zero_division=0)

print("\nOptimisation du réseau de neurones :")
print(f"Weighted F1 Score pour le réseau de neurones après optimisation : {weighted_f1_nn:.3f}")
print("Classification Report :\n", report_nn)



Fitting 10 folds for each of 1440 candidates, totalling 14400 fits


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
3600 fits failed out of a total of 14400.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3600 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\loc

Meilleurs hyperparamètres : {'C': 1, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs', 'tol': 0.001, 'warm_start': True}

Optimisation de la régression logistique :
Weighted F1 Score après optimisation : 0.777
Classification Report :
               precision    recall  f1-score   support

           0       0.65      0.44      0.52       259
           1       0.82      0.92      0.87       741

    accuracy                           0.79      1000
   macro avg       0.73      0.68      0.69      1000
weighted avg       0.78      0.79      0.78      1000



C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1583: UserWarning: Note that pos_label (set to 'Pris en compte') is ignored when average != 'binary' (got 'weighted'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(


In [28]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées
y = train_df['cal_réponse_signalement']  # Variable cible

# Plage de valeurs pour n_components à tester
n_components_range = list(range(50, 301, 50))
variance_threshold = 0.85  # Objectif de variance cumulative souhaitée (ajustable entre 0.80 et 0.90)
best_n_components = None
best_explained_variance = 0

# Recherche du meilleur n_components pour capturer la variance désirée
for n_components in n_components_range:
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)

    # Calcul de la variance cumulée
    cumulative_variance = svd.explained_variance_ratio_.sum()
    
    if cumulative_variance >= variance_threshold and cumulative_variance > best_explained_variance:
        best_n_components = n_components
        best_explained_variance = cumulative_variance

print(f"\nMeilleure valeur pour n_components : {best_n_components} avec variance cumulée : {best_explained_variance:.4f}")

# Appliquer TruncatedSVD avec le meilleur n_components
svd = TruncatedSVD(n_components=best_n_components, random_state=42)
X_reduced = svd.fit_transform(X)

# Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
scaler = MinMaxScaler()
X_reduced = scaler.fit_transform(X_reduced)
final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(best_n_components)])

# Séparation des données (caractéristiques réduites et cible)
X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

# Création des modèles de classification
models = {
    #'Multinomial Naive Bayes': MultinomialNB(),
    #'Bernoulli Naive Bayes': BernoulliNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K Nearest Neighbors': KNeighborsClassifier(),
    #'Decision Tree': DecisionTreeClassifier(),
    #'Random Forest': RandomForestClassifier()
}

# Liste pour stocker les résultats du weighted F1 score par modèle
f1_scores = []

# Évaluation des modèles avec les données réduites
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average='weighted',pos_label="Pris en compte")
    report = classification_report(y_test, y_pred, zero_division=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)

    # Enregistrer le weighted F1 score pour la visualisation
    f1_scores.append(weighted_f1)

    # Affichage des résultats
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Weighted F1 Score: {weighted_f1:.2f}")
    print(f"Classification Report:\n{report}")
    print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
    print('-' * 50)

# Visualisation de l'évolution du weighted F1 score par classifieur
plt.figure(figsize=(10, 6))
plt.bar(models.keys(), f1_scores, color='skyblue')
plt.title('Weighted F1 Score par Classifieur')
plt.xlabel('Classifieurs')
plt.ylabel('Weighted F1 Score')
plt.xticks(rotation=45)
plt.show()

Les prédictions ont été sauvegardées dans 'test_predictions.csv'.


In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Utiliser final_df sans réduction de dimension comme ensemble de données
X = final_train_df  # Les données vectorisées sans réduction de dimension
y = train_df['cal_réponse_signalement']  # Variable cible

# Assurer que les dimensions de X et y correspondent
assert X.shape[0] == y.shape[0], "Les dimensions de X et y ne correspondent pas"

# Séparation des données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définition de l'espace des hyperparamètres pour la régression logistique
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  # Essai de différentes valeurs pour le paramètre de régularisation
    'solver': ['liblinear', 'lbfgs'],  # Différents solveurs possibles pour optimiser
    'penalty': ['l2'],  # 'l1' est disponible uniquement pour le solver 'liblinear'
    'max_iter': [100, 200, 500],  # Tester différents nombres d'itérations
    'dual': [True, False],  # Formulation duale
    'fit_intercept': [True, False],  # Ajouter un terme de biais au modèle
    'class_weight': [None, 'balanced'],  # Gérer les déséquilibres des classes
    'tol': [1e-4, 1e-3, 1e-2],  # Tolérance pour le critère d'arrêt
    'warm_start': [True, False]  # Utiliser la solution de l'entraînement précédent
}



# Initialisation du modèle de régression logistique
log_reg = LogisticRegression()

# Utilisation de GridSearchCV pour la recherche d'hyperparamètres
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=10, scoring='f1_weighted', n_jobs=-1, verbose=3)

# Exécution de GridSearchCV
grid_search.fit(X_train, y_train)

# Affichage des meilleurs hyperparamètres
print("Meilleurs hyperparamètres :", grid_search.best_params_)

# Entraînement du modèle avec les meilleurs hyperparamètres
best_log_reg = grid_search.best_estimator_
best_log_reg.fit(X_train, y_train)
y_pred = best_log_reg.predict(X_test)

# Calcul du weighted F1 score et affichage du rapport de classification
weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
report = classification_report(y_test, y_pred, zero_division=0)

print("\nOptimisation de la régression logistique :")
print(f"Weighted F1 Score après optimisation : {weighted_f1:.3f}")
print("Classification Report :\n", report)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Utiliser final_df, votre DataFrame vectorisé
X = final_df  # Les données vectorisées sans NaN
y = train_df['cal_réponse_signalement']  # Variable cible sans NaN

# Assurer que les dimensions de X et y correspondent
assert X.shape[0] == y.shape[0], "Les dimensions de X et y ne correspondent pas"

# Liste des valeurs de n_components à tester manuellement
n_components_to_test = [300, 325, 400, 500, 600]

# Dictionnaire pour stocker les scores pour chaque valeur de n_components
results = {model: [] for model in ['Logistic Regression', 'Neural Network']}

# Fonction pour créer le modèle de réseau de neurones
def create_nn_model(input_dim):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Boucle pour tester chaque valeur de n_components
for n_components in n_components_to_test:
    print(f"Testing n_components={n_components}")
    
    # Appliquer TruncatedSVD
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)
    
    # Afficher la variance expliquée par chaque composante
    cumulative_variance = svd.explained_variance_ratio_.sum()
    print(f"Cumulative variance for n_components={n_components}: {cumulative_variance:.4f}")

    # Normaliser les données après réduction avec MinMaxScaler pour éviter les valeurs négatives
    scaler = MinMaxScaler()
    X_reduced = scaler.fit_transform(X_reduced)
    final_df_reduced = pd.DataFrame(X_reduced, columns=[f"svd_component_{i+1}" for i in range(n_components)])

    # Séparation des données (caractéristiques réduites et cible)
    X_train, X_test, y_train, y_test = train_test_split(final_df_reduced, y, test_size=0.2, random_state=42)

    # Création des modèles de classification
    models = {
        #'Logistic Regression': LogisticRegression(max_iter=1000),
        'Neural Network': create_nn_model(X_train.shape[1]),
    }

    # Évaluation des modèles avec les données réduites
    for name, model in models.items():
        if name == 'Neural Network':
            # Entraînement du modèle de réseau de neurones
            model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
            y_pred = (model.predict(X_test) > 0.5).astype("int32")
            accuracy = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
            results[name].append((n_components, weighted_f1))
        else:
            # Entraînement des autres modèles
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            weighted_f1 = f1_score(y_test, y_pred, average='weighted', pos_label="Pris en compte")
            results[name].append((n_components, weighted_f1))
            cv_scores = cross_val_score(model, X_train, y_train, cv=5)
        
        # Affichage des résultats
        print(f"Model: {name}")
        print(f"Accuracy: {accuracy:.2f}")
        print(f"Weighted F1 Score: {weighted_f1:.3f}")
        if name != 'Neural Network':
            report = classification_report(y_test, y_pred, zero_division=0)
            print(f"Classification Report:\n{report}")
            print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.2f}")
        print('-' * 50)

# Visualisation des résultats pour chaque modèle
plt.figure(figsize=(12, 8))
for model, scores in results.items():
    n_components, f1_scores = zip(*scores)
    plt.plot(n_components, f1_scores, marker='o', label=model)
    for x, y in zip(n_components, f1_scores):
        plt.text(x, y, f'{y:.3f}', fontsize=9, ha='right')

plt.title('Weighted F1 Score par Modèle en fonction de n_components')
plt.xlabel('n_components')
plt.ylabel('Weighted F1 Score')
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
test_predic = pd.read_csv('C:/Users/gauth/Desktop/Projet-Code/Python/IA/Projet DATA-Mining/test_predictions.csv')
test_predic['cal_réponse_signalement'] = test_predic['cal_réponse_signalement'].map({1: 'Pris en compte', 0: 'Rejete (hors specs)'})
test_predic.to_csv('test_predictions.csv', index=False)

In [ ]:
# Télécharger les stopwords français si nécessaire
nltk.download('stopwords')

# Initialiser le modèle spaCy pour la lemmatisation
nlp = spacy.load('fr_core_news_sm')

# Définir les colonnes textuelles et catégorielles
text_columns = ['boamp_theme_boamp', 'boamp_intitule', 'texte_annonce']
categorical_columns = ['cal_mot_rang_1', 'cal_mot_rang_2', 'boamp_libelle_annonce', 'boamp_statut_annonce', 'cal_theme_signalement', 'boamp_num_departement_diffusion']

# Ajouter des mots vides supplémentaires
additional_stop_words = set([
    "de","la", "le", "les", 
    "des", "du", "un", "une", "en", "et", 
    "à", "au", "aux", "par", "pour", "sur", 
    "dans", "avec", "ce", "cette", "ces", "mon", 
    "ma", "mes", "ton", "ta", "tes", "son", "sa", 
    "ses", "notre", "nos", "votre", "vos", "leur", 
    "leurs", "ne", "pas", "plus", "moins", "ou", 
    "si", "oui", "non", "je", "tu", "il", "elle", 
    "nous", "vous", "ils", "elles", "me", "te", 
    "se", "moi", "toi", "lui", "leur", "y", "en",
    "d'", "l'","s'", "c'", "n'", "j'", "t'", "m'", "qu'", "l'"    
])

stop_words = set(stopwords.words("french")).union(additional_stop_words)

# Fonction de nettoyage et de lemmatisation des textes
def clean_text(text):
    if isinstance(text, float):
        return ''

    # Suppression des caractères spéciaux et de la ponctuation (conservation des lettres avec accents et apostrophes)
    text = re.sub(r'[^\w\s\']', '', text)

    # Conversion en minuscule
    text = text.lower()

    # Traitement du texte avec spaCy pour lemmatisation
    doc = nlp(text)

    # Suppression des mots vides
    lemmatized_tokens = [token.lemma_ for token in doc if token.text not in stop_words]

    # Joindre les tokens en une seule chaîne
    cleaned_text = ' '.join(lemmatized_tokens)

    return cleaned_text

# Étape 1 : Appliquer le nettoyage et la lemmatisation aux colonnes textuelles
for col in text_columns:
    train_df[col] = train_df[col].apply(lambda x: clean_text(x))

# Étape 2 : Vectorisation des colonnes textuelles avec TF-IDF (sans réduction des dimensions)
vectorized_dfs = {}
for col in text_columns:
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))  # Inclut unigrams et bigrams
    vectorized_df = pd.DataFrame(
        vectorizer.fit_transform(train_df[col]).toarray(),
        columns=[f"{col}_{feat}" for feat in vectorizer.get_feature_names_out()]
    )
    vectorized_dfs[col] = vectorized_df

# Étape 3 : Encodage des colonnes catégorielles avec OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_encoded = pd.DataFrame(
    encoder.fit_transform(train_df[categorical_columns]),
    columns=encoder.get_feature_names_out(categorical_columns)
)

# Étape 4 : Concaténer toutes les colonnes vectorisées et encodées
concat1_df = pd.concat([*vectorized_dfs.values(), categorical_encoded], axis=1)

# Affichage d'un échantillon des données finales
print(concat1_df.head())
